# Model 1 — Independent ResNet18 Baseline for LADI-v2

# LADI-v2 independent Kaggle notebook

This notebook is **fully self-contained**. It does not depend on outputs from notebook `00`, `01`, or any other notebook.

It will:

1. install/check requirements,
2. load the LADI-v2 dataset directly from Hugging Face,
3. prepare train/validation/test dataloaders,
4. train the selected model,
5. tune validation thresholds,
6. evaluate on validation and test,
7. save checkpoint, predictions, metrics, and configuration under `/kaggle/working/`.

**Kaggle setting:** GPU P100, Internet ON.

The default Hugging Face branch loads the compact `v2a` label set with resized images. If you only want a smoke test, set `LIMIT_TRAIN`, `LIMIT_VAL`, and `LIMIT_TEST` below.


**P100 CUDA compatibility note:** this patched notebook pins PyTorch `2.4.1` + torchvision `0.19.1` with CUDA `11.8` to avoid `cudaErrorNoKernelImageForDevice` on Tesla P100. Run from a fresh Kaggle session with Internet ON.


In [1]:
# ============================================================
# 0. P100-compatible requirements installation
# ============================================================
# IMPORTANT FOR KAGGLE P100:
# New Kaggle/PyTorch CUDA builds may not include kernels for Tesla P100 (sm_60),
# which causes: "CUDA error: no kernel image is available for execution on the device".
# This cell pins a CUDA 11.8 PyTorch build that is usually safe for P100.
# Run this notebook from a fresh Kaggle session with Internet ON.

import sys, subprocess, os

INSTALL_P100_TORCH_FIX = True

if INSTALL_P100_TORCH_FIX:
    print("Installing P100-compatible PyTorch/torchvision build...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
        "torch==2.4.1", "torchvision==0.19.1",
        "--index-url", "https://download.pytorch.org/whl/cu118"
    ])

print("Installing remaining lightweight dependencies...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets", "scikit-learn", "pandas", "matplotlib", "tqdm", "pillow==11.3.0"
])

# Do not import torch before the installation above.
# If you already ran the old notebook and got the CUDA error, choose:
# Kaggle menu → Session options → Restart session / Factory reset runtime,
# then run this patched notebook from the first cell.


Installing P100-compatible PyTorch/torchvision build...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 MB 258.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 272.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 372.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 359.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 357.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 357.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 323.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 358.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 296.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 371.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 384.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2026.4.0 which is incompatible.
datasets 4.8.3 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, b

Installing remaining lightweight dependencies...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.4.1+cu118 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


0

In [2]:

# ============================================================
# 1. Imports, configuration, reproducibility
# ============================================================
import os, json, math, random, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------- P100 / CUDA sanity check --------------------
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    arch_list = torch.cuda.get_arch_list()
    print("GPU:", gpu_name)
    print("Compute capability:", cap)
    print("Torch CUDA arch list:", arch_list)
    if cap == (6, 0) and "sm_60" not in arch_list:
        raise RuntimeError(
            "This PyTorch build does not include sm_60 kernels for Tesla P100. "
            "Restart the Kaggle session and rerun the first installation cell."
        )
    # Small CUDA operation to fail early if the installation is incompatible.
    _x = torch.randn(8, 8, device="cuda")
    _y = (_x @ _x.T).sum()
    torch.cuda.synchronize()
    print("CUDA sanity check passed.")


# -------------------- editable config --------------------
DATASET_NAME = "MITLL/LADI-v2-dataset"
OUTPUT_ROOT = Path("/kaggle/working/ladi_independent_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# For a fast smoke test, set e.g. LIMIT_TRAIN=512, LIMIT_VAL=128, LIMIT_TEST=128.
# For the real experiment, keep them as None.
LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

IMAGE_SIZE = 320       # P100-safe. If OOM: try 288 or 224.
BATCH_SIZE = 16        # Dynamic GCN notebook may override this to 8 or 12.
NUM_WORKERS = 2
NUM_EPOCHS = 12        # Increase to 20-30 for final runs.
LR = 2e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
PATIENCE = 4

LABEL_COLS = [
    "bridges_any",
    "buildings_any",
    "buildings_affected_or_greater",
    "buildings_minor_or_greater",
    "debris_any",
    "flooding_any",
    "flooding_structures",
    "roads_any",
    "roads_damage",
    "trees_any",
    "trees_damage",
    "water_any",
]
NUM_LABELS = len(LABEL_COLS)
print("Labels:", NUM_LABELS, LABEL_COLS)

# RUN_NAME is set by each notebook.


# -------------------- model-specific config --------------------
RUN_NAME = "model_01_resnet18_baseline_independent"
BACKBONE_NAME = "resnet18"
DROPOUT = 0.25
CONFIG = {
    "run_name": RUN_NAME,
    "model": "CNNBaseline",
    "backbone": BACKBONE_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "limit_train": LIMIT_TRAIN,
    "limit_val": LIMIT_VAL,
    "limit_test": LIMIT_TEST,
    "label_cols": LABEL_COLS,
}
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)


Device: cuda
GPU: Tesla P100-PCIE-16GB
Torch: 2.4.1+cu118 CUDA build: 11.8
GPU: Tesla P100-PCIE-16GB
Compute capability: (6, 0)
Torch CUDA arch list: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_37', 'sm_90']
CUDA sanity check passed.
Labels: 12 ['bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any']


In [3]:

# ============================================================
# 2. Dataset loading and dataloaders: included inside this notebook
# ============================================================
def subset_hf_dataset(ds_split, limit):
    if limit is None:
        return ds_split
    limit = min(limit, len(ds_split))
    return ds_split.select(range(limit))

print("Loading LADI-v2 from Hugging Face...")
# Default config loads v2a with resized images according to the HF dataset card.
ds = load_dataset(DATASET_NAME)
print(ds)

train_hf = subset_hf_dataset(ds["train"], LIMIT_TRAIN)
val_hf = subset_hf_dataset(ds["validation"], LIMIT_VAL)
test_hf = subset_hf_dataset(ds["test"], LIMIT_TEST)

print("Split sizes:", len(train_hf), len(val_hf), len(test_hf))

def get_labels_matrix(hf_split):
    rows = []
    for ex in tqdm(hf_split, desc="Collecting labels"):
        rows.append([float(bool(ex[c])) for c in LABEL_COLS])
    return np.asarray(rows, dtype=np.float32)

Y_train_np = get_labels_matrix(train_hf)
Y_val_np = get_labels_matrix(val_hf)
Y_test_np = get_labels_matrix(test_hf)

print("Training label prevalence:")
for name, prev in zip(LABEL_COLS, Y_train_np.mean(axis=0)):
    print(f"  {name:34s}: {prev:.4f}")

# ImageNet transforms; CNN pretrained weights expect this normalization.
train_tfms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.35),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class LADIMultiLabelDataset(Dataset):
    def __init__(self, hf_split, transform, label_cols):
        self.hf_split = hf_split
        self.transform = transform
        self.label_cols = label_cols
    def __len__(self):
        return len(self.hf_split)
    def __getitem__(self, idx):
        ex = self.hf_split[int(idx)]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert("RGB")
        x = self.transform(img)
        y = torch.tensor([float(bool(ex[c])) for c in self.label_cols], dtype=torch.float32)
        return x, y, int(idx)

train_loader = DataLoader(
    LADIMultiLabelDataset(train_hf, train_tfms, LABEL_COLS),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    LADIMultiLabelDataset(val_hf, eval_tfms, LABEL_COLS),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    LADIMultiLabelDataset(test_hf, eval_tfms, LABEL_COLS),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

# Class imbalance weights for BCEWithLogitsLoss.
pos = Y_train_np.sum(axis=0)
neg = len(Y_train_np) - pos
pos_weight_np = neg / np.maximum(pos, 1.0)
pos_weight_np = np.clip(pos_weight_np, 1.0, 20.0).astype(np.float32)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=device)
print("pos_weight:", dict(zip(LABEL_COLS, pos_weight_np.round(2))))

# ============================================================
# 3. Metrics, threshold tuning, train/evaluate helpers
# ============================================================
def safe_average_precision(y_true, y_score):
    vals = []
    for j in range(y_true.shape[1]):
        if y_true[:, j].sum() == 0:
            continue
        vals.append(average_precision_score(y_true[:, j], y_score[:, j]))
    return float(np.mean(vals)) if vals else 0.0

def multilabel_metrics(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    y_pred = (y_prob >= thresholds[None, :]).astype(np.int32)
    out = {
        "macro_ap": safe_average_precision(y_true, y_prob),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    per_label = []
    for j, name in enumerate(LABEL_COLS):
        ap = average_precision_score(y_true[:, j], y_prob[:, j]) if y_true[:, j].sum() > 0 else 0.0
        per_label.append({
            "label": name,
            "prevalence": float(y_true[:, j].mean()),
            "ap": float(ap),
            "f1": float(f1_score(y_true[:, j], y_pred[:, j], zero_division=0)),
            "threshold": float(thresholds[j]),
        })
    return out, pd.DataFrame(per_label)

def tune_thresholds(y_true, y_prob):
    grid = np.arange(0.05, 0.96, 0.05)
    th = np.zeros(y_true.shape[1], dtype=np.float32)
    for j in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            pred = (y_prob[:, j] >= t).astype(np.int32)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_t, best_f1 = float(t), float(f1)
        th[j] = best_t
    return th

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    probs, targets, indices = [], [], []
    for x, y, idx in tqdm(loader, desc="Predict", leave=False):
        x = x.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP and device.type == "cuda"):
            logits = model(x)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        targets.append(y.numpy())
        indices.append(np.asarray(idx))
    return np.vstack(probs), np.vstack(targets), np.concatenate(indices)

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    n = 0
    for x, y, _ in tqdm(loader, desc="Train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP and device.type == "cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        bs = x.size(0)
        total_loss += float(loss.detach().cpu()) * bs
        n += bs
    return total_loss / max(n, 1)

def save_predictions_csv(path, probs, y_true, indices, thresholds):
    rows = []
    y_pred = (probs >= thresholds[None, :]).astype(np.int32)
    for i in range(len(probs)):
        row = {"index": int(indices[i])}
        for j, name in enumerate(LABEL_COLS):
            row[f"true_{name}"] = int(y_true[i, j])
            row[f"prob_{name}"] = float(probs[i, j])
            row[f"pred_{name}"] = int(y_pred[i, j])
        rows.append(row)
    pd.DataFrame(rows).to_csv(path, index=False)

def run_training(model, run_dir):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = GradScaler(enabled=USE_AMP and device.type == "cuda")

    best_ap = -1.0
    best_epoch = -1
    bad_epochs = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        scheduler.step()
        val_prob, val_true, val_idx = predict_loader(model, val_loader)
        val_metrics, _ = multilabel_metrics(val_true, val_prob, thresholds=None)
        print({"train_loss": round(loss, 4), **{k: round(v, 4) for k, v in val_metrics.items()}})
        history.append({"epoch": epoch, "train_loss": loss, **val_metrics})

        if val_metrics["macro_ap"] > best_ap:
            best_ap = val_metrics["macro_ap"]
            best_epoch = epoch
            bad_epochs = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "label_cols": LABEL_COLS,
                "config": CONFIG,
                "best_val_macro_ap": best_ap,
                "best_epoch": best_epoch,
            }, run_dir / "best_model.pt")
            print("Saved new best checkpoint.")
        else:
            bad_epochs += 1
            print(f"No improvement. bad_epochs={bad_epochs}/{PATIENCE}")
            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    checkpoint = torch.load(run_dir / "best_model.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    # Tune thresholds on validation only.
    val_prob, val_true, val_idx = predict_loader(model, val_loader)
    thresholds = tune_thresholds(val_true, val_prob)
    val_metrics, val_per_label = multilabel_metrics(val_true, val_prob, thresholds)

    test_prob, test_true, test_idx = predict_loader(model, test_loader)
    test_metrics, test_per_label = multilabel_metrics(test_true, test_prob, thresholds)

    save_predictions_csv(run_dir / "val_predictions.csv", val_prob, val_true, val_idx, thresholds)
    save_predictions_csv(run_dir / "test_predictions.csv", test_prob, test_true, test_idx, thresholds)
    val_per_label.to_csv(run_dir / "val_per_label_metrics.csv", index=False)
    test_per_label.to_csv(run_dir / "test_per_label_metrics.csv", index=False)

    metrics = {
        "run_name": RUN_NAME,
        "best_epoch": int(best_epoch),
        "best_val_macro_ap_checkpoint": float(best_ap),
        "val_metrics_threshold_tuned": val_metrics,
        "test_metrics_threshold_tuned": test_metrics,
        "thresholds": {name: float(t) for name, t in zip(LABEL_COLS, thresholds)},
    }
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    with open(run_dir / "config.json", "w") as f:
        json.dump(CONFIG, f, indent=2)

    print("\nFinal validation metrics:", val_metrics)
    print("Final test metrics:", test_metrics)
    print("Saved outputs to:", run_dir)
    return metrics


Loading LADI-v2 from Hugging Face...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

...

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/40 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any'],
        num_rows: 8030
    })
    validation: Dataset({
        features: ['image', 'bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any'],
        num_rows: 890
    })
    test: Dataset({
        features: ['image', 'bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any'],
        num_rows: 1049
    })
})
Split sizes: 8030 890 1049


Training label prevalence:
  bridges_any                       : 0.0716
  buildings_any                     : 0.5938
  buildings_affected_or_greater     : 0.1853
  buildings_minor_or_greater        : 0.0593
  debris_any                        : 0.0839
  flooding_any                      : 0.1812
  flooding_structures               : 0.0552
  roads_any                         : 0.6585
  roads_damage                      : 0.0448
  trees_any                         : 0.9045
  trees_damage                      : 0.2446
  water_any                         : 0.6585
pos_weight: {'bridges_any': np.float32(12.97), 'buildings_any': np.float32(1.0), 'buildings_affected_or_greater': np.float32(4.4), 'buildings_minor_or_greater': np.float32(15.87), 'debris_any': np.float32(10.91), 'flooding_any': np.float32(4.52), 'flooding_structures': np.float32(17.13), 'roads_any': np.float32(1.0), 'roads_damage': np.float32(20.0), 'trees_any': np.float32(1.0), 'trees_damage': np.float32(3.09), 'water_any': np.

In [4]:

# ============================================================
# 4. Model definition: CNN multi-label baseline
# ============================================================
def build_backbone(backbone_name="resnet18", pretrained=True):
    if backbone_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        m = models.resnet18(weights=weights)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
        return m, feat_dim
    elif backbone_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        m = models.resnet50(weights=weights)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
        return m, feat_dim
    elif backbone_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b0(weights=weights)
        feat_dim = m.classifier[1].in_features
        m.classifier = nn.Identity()
        return m, feat_dim
    elif backbone_name == "efficientnet_b2":
        weights = models.EfficientNet_B2_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b2(weights=weights)
        feat_dim = m.classifier[1].in_features
        m.classifier = nn.Identity()
        return m, feat_dim
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

class CNNBaseline(nn.Module):
    def __init__(self, backbone_name, num_labels, dropout=0.25, pretrained=True):
        super().__init__()
        self.backbone, feat_dim = build_backbone(backbone_name, pretrained=pretrained)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_labels)
        )
    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)

model = CNNBaseline(BACKBONE_NAME, NUM_LABELS, dropout=DROPOUT, pretrained=True).to(device)
print(model.__class__.__name__, "with", BACKBONE_NAME)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 238MB/s]


CNNBaseline with resnet18
Trainable parameters: 11182668


In [5]:

# ============================================================
# Final cell: train, validate, test, and save outputs
# ============================================================
print("Run name:", RUN_NAME)
print("Output directory:", RUN_DIR)
metrics = run_training(model, RUN_DIR)
metrics


Run name: model_01_resnet18_baseline_independent
Output directory: /kaggle/working/ladi_independent_runs/model_01_resnet18_baseline_independent

Epoch 1/12


Train:   0%|          | 0/502 [00:00<?, ?it/s]

Predict:   0%|          | 0/56 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78521d3f1f80>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78521d3f1f80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1477, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1423, in _shutdown_workers
    self._pin_memory_thread.join()
  File "/usr/lib/python3.12/threading.py", line 1146, in join
    raise RuntimeError("cannot join current thread")
RuntimeError: cannot join current thread

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1477, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1460, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/

{'train_loss': 0.7355, 'macro_ap': 0.6993, 'macro_f1': 0.6136, 'micro_f1': 0.7726, 'macro_precision': 0.5544, 'macro_recall': 0.7778}
Saved new best checkpoint.

Epoch 2/12


...

{'train_loss': 0.1366, 'macro_ap': 0.7914, 'macro_f1': 0.7412, 'micro_f1': 0.8651, 'macro_precision': 0.7523, 'macro_recall': 0.7323}
Saved new best checkpoint.


Predict:   0%|          | 0/56 [00:00<?, ?it/s]

Predict:   0%|          | 0/66 [00:00<?, ?it/s]


Final validation metrics: {'macro_ap': 0.7914027776199423, 'macro_f1': 0.7505832800987232, 'micro_f1': 0.8714519245058702, 'macro_precision': 0.759024281608581, 'macro_recall': 0.7460606154280658}
Final test metrics: {'macro_ap': 0.5366288050887856, 'macro_f1': 0.5383647273528529, 'micro_f1': 0.8170028818443804, 'macro_precision': 0.5247214606597572, 'macro_recall': 0.5871361800427922}
Saved outputs to: /kaggle/working/ladi_independent_runs/model_01_resnet18_baseline_independent


{'run_name': 'model_01_resnet18_baseline_independent',
 'best_epoch': 12,
 'best_val_macro_ap_checkpoint': 0.7914027776199423,
 'val_metrics_threshold_tuned': {'macro_ap': 0.7914027776199423,
  'macro_f1': 0.7505832800987232,
  'micro_f1': 0.8714519245058702,
  'macro_precision': 0.759024281608581,
  'macro_recall': 0.7460606154280658},
 'test_metrics_threshold_tuned': {'macro_ap': 0.5366288050887856,
  'macro_f1': 0.5383647273528529,
  'micro_f1': 0.8170028818443804,
  'macro_precision': 0.5247214606597572,
  'macro_recall': 0.5871361800427922},
 'thresholds': {'bridges_any': 0.550000011920929,
  'buildings_any': 0.25,
  'buildings_affected_or_greater': 0.550000011920929,
  'buildings_minor_or_greater': 0.550000011920929,
  'debris_any': 0.3499999940395355,
  'flooding_any': 0.6000000238418579,
  'flooding_structures': 0.5,
  'roads_any': 0.44999998807907104,
  'roads_damage': 0.44999998807907104,
  'trees_any': 0.44999998807907104,
  'trees_damage': 0.44999998807907104,
  'water_any'

In [6]:
df=pd.read_csv("/kaggle/working/ladi_independent_runs/model_01_resnet18_baseline_independent/test_per_label_metrics.csv")
df

,label,prevalence,ap,f1,threshold
0,bridges_any,0.050524,0.421361,0.390805,0.55
1,buildings_any,0.724500,0.973822,0.926316,0.25
2,buildings_affected_or_greater,0.045758,0.436267,0.522388,0.55
3,buildings_minor_or_greater,0.020972,0.487929,0.530612,0.55
4,debris_any,0.028599,0.501876,0.450980,0.35
5,flooding_any,0.078170,0.438248,0.473373,0.60
6,flooding_structures,0.011439,0.167951,0.210526,0.50
7,roads_any,0.806482,0.956334,0.859660,0.45
8,roads_damage,0.010486,0.048382,0.133333,0.45
9,trees_any,0.931363,0.995501,0.954186,0.45


In [7]:
df1=pd.read_csv("/kaggle/working/ladi_independent_runs/model_01_resnet18_baseline_independent/test_predictions.csv")
df1

,index,true_bridges_any,prob_bridges_any,pred_bridges_any,true_buildings_any,prob_buildings_any,pred_buildings_any,true_buildings_affected_or_greater,prob_buildings_affected_or_greater,pred_buildings_affected_or_greater,...,pred_roads_damage,true_trees_any,prob_trees_any,pred_trees_any,true_trees_damage,prob_trees_damage,pred_trees_damage,true_water_any,prob_water_any,pred_water_any
0,0,0,0.011734,0,1,0.988281,1,1,0.164673,0,...,0,1,0.966797,1,1,0.131226,0,0,0.527832,1
1,1,0,0.016403,0,1,0.995605,1,1,0.679199,1,...,0,1,0.966797,1,1,0.541992,1,0,0.173828,0
2,2,0,0.002581,0,0,0.048767,0,0,0.001008,0,...,0,1,0.998047,1,1,0.094482,0,1,0.998047,1
3,3,0,0.002132,0,1,0.988770,1,1,0.948730,1,...,0,1,0.986816,1,1,0.804199,1,0,0.189087,0
4,4,0,0.002901,0,1,0.966797,1,1,0.372070,0,...,0,1,0.994141,1,1,0.753906,1,0,0.112183,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,1044,0,0.000238,0,0,0.607910,1,0,0.057709,0,...,0,1,1.000000,1,1,0.967773,1,1,0.996582,1
1045,1045,0,0.008575,0,1,0.905273,1,0,0.250000,0,...,0,1,0.999512,1,1,0.920410,1,1,0.998047,1
1046,1046,0,0.027908,0,1,0.981934,1,0,0.782227,1,...,0,1,0.972656,1,0,0.292480,0,1,0.686035,1
1047,1047,0,0.002481,0,0,0.011162,0,0,0.011330,0,...,0,1,0.998535,1,1,0.992188,1,1,0.999512,1
